# Dynamic Programming approach for solving TSP (Bellman-Held-Karp Algorithm) (Exact Solutions)

*Recurrence Relation: dp[S][i]=min(dp[S−{i}][j]+dist(j,i)) for all j∈S,j!=i*
<br>
*Time Complexity: O(n^2 * 2^n)*
<br>
*Space Complexity: O(n * 2^n)*

In [1]:
import itertools
import numpy as np
import time

In [2]:
class TSPBellmanHeldKarpSolver:
    def __init__(self, distanceMatrix):
        """
        Initialize the solver with the distance matrix.

        :param distanceMatrix: 2D list containing distances between cities
        """
        self.distanceMatrix = np.array(distanceMatrix)
        self.numCities = len(distanceMatrix)
        self.dp = {}  # Dictionary to store the subproblem solutions
    
    def solve(self):
        """
        Solves the TSP using the Bellman-Held-Karp algorithm.

        :return: Tuple containing the best route and the minimum cost
        """
        start_time = time.time()
        # Populate the dp table using dynamic programming
        self.dynamicProgramming()

        # Find the minimum cost by completing the tour (returning to city 0)
        minCost, bestRoute = self.findOptimalRoute()
        end_time = time.time()

        print(f"Runtime: {end_time - start_time:.3f} seconds")
        
        return bestRoute, minCost

    def dynamicProgramming(self):
        """
        Populates the dynamic programming table.
        """
        # Initialize the base case: starting at city 0
        for i in range(1, self.numCities):
            self.dp[(1 << i, i)] = self.distanceMatrix[0, i]

        # Loop over the number of cities in the subset (2 to numCities)
        for subsetSize in range(2, self.numCities):
            for subset in itertools.combinations(range(1, self.numCities), subsetSize):
                subsetMask = self.getSubsetMask(subset)
                # Iterate over all cities in the subset
                for destinationCity in subset:
                    prevMask = subsetMask ^ (1 << destinationCity)
                    minCost = float('inf')

                    # Find the best previous step (coming from another city)
                    for k in subset:
                        if k == destinationCity:
                            continue
                        cost = self.dp[(prevMask, k)] + self.distanceMatrix[k, destinationCity]
                        minCost = min(minCost, cost)

                    # Store the result for the current subset and destination city
                    self.dp[(subsetMask, destinationCity)] = minCost

    def findOptimalRoute(self):
        """
        Finds the minimum cost by completing the cycle back to city 0.

        :return: Minimum cost and the best route
        """
        finalMask = (1 << self.numCities) - 2  # All cities except the starting city
        minCost = float('inf')
        lastCity = -1

        # Find the best last city to return to city 0
        for i in range(1, self.numCities):
            cost = self.dp[(finalMask, i)] + self.distanceMatrix[i, 0]
            if cost < minCost:
                minCost = cost
                lastCity = i

        # Reconstruct the optimal route
        bestRoute = self.reconstructRoute(lastCity, finalMask)
        return minCost, bestRoute

    def reconstructRoute(self, lastCity, finalMask):
        """
        Reconstructs the best route by backtracking through the dp table.

        :param lastCity: The last city in the optimal route before returning to city 0
        :param finalMask: The bitmask representing all visited cities
        :return: List representing the optimal route
        """
        route = [0]  # Start from city 0
        currentMask = finalMask
        currentCity = lastCity

        while currentCity != -1:
            route.append(currentCity)
            nextCity = -1
            # Backtrack to the previous city
            for i in range(1, self.numCities):
                if currentMask & (1 << i) and i != currentCity:
                    if self.dp[(currentMask, currentCity)] == self.dp[(currentMask ^ (1 << currentCity), i)] + self.distanceMatrix[i, currentCity]:
                        nextCity = i
                        break
            currentMask ^= (1 << currentCity)
            currentCity = nextCity

        return route[::-1]  # Reverse the route to get the correct order

    def getSubsetMask(self, subset):
        """
        Converts a subset of cities into a bitmask.

        :param subset: A tuple of city indices
        :return: An integer bitmask where 1 represents that a city is in the subset
        """
        mask = 0
        for city in subset:
            mask |= (1 << city)
        return mask

In [3]:
class TSPApp:
    def __init__(self):
        """
        Initialize the application.
        """
        self.distanceMatrix = None
        self.solver = None
    
    def setDistanceMatrix(self, distanceMatrix):
        """
        Set the distance matrix and initialize the solver.

        :param distanceMatrix: 2D list containing distances between cities
        """
        self.distanceMatrix = distanceMatrix
        self.solver = TSPBellmanHeldKarpSolver(distanceMatrix)
    
    def run(self):
        """
        Run the TSP solver and display the results.
        """
        if self.solver is None:
            raise Exception("Distance matrix is not set.")
        
        bestRoute, minCost = self.solver.solve()
        self.displayResults(bestRoute, minCost)
    
    def displayResults(self, bestRoute, minCost):
        """
        Display the best route and minimum cost.

        :param bestRoute: Best route found
        :param minCost: Minimum cost of the route
        """
        print("Best Route:", bestRoute)
        print("Minimum Cost:", minCost)
        self.printPath(bestRoute)
    
    def generateRandomGraph(self, n, max_distance=100):
        """
        Generate a random distance matrix for n cities.

        :param n: Number of cities
        :param max_distance: Maximum distance between any two cities
        :return: 2D list representing the distance matrix
        """

        distanceMatrix = np.random.randint(1, max_distance + 1, size=(n, n))
        np.fill_diagonal(distanceMatrix, 0)
        return distanceMatrix
    
    def printPath(self, path):
        """
        Print the path in list or array form.

        :param path: List representing the path (e.g., [0, 2, 3, 1, 0])
        """
        path_array = np.array(path)
        print("Path in array form:", path_array)

In [6]:
n = 30  # Number of cities
tspApp = TSPApp()
distanceMatrix = tspApp.generateRandomGraph(n)
tspApp.setDistanceMatrix(distanceMatrix)

print("Distance Matrix:\n", distanceMatrix)

Distance Matrix:
 [[  0  67  98  80  37  93  29  46  98  13   7  56  77  49  26  95  88  23
   68  19  55  83  62  51  53  28  23   7  17  47]
 [ 14   0  32  39  41  44  90  82  80  26  82   1   8  45  49  73  52  96
   86  26  29  45  37  94  33  86  20  72  75  23]
 [ 45  28   0  93  76  31  39  97  17  43  65  75  75  22   7  16  39  95
   10  77   1  36  73  46  23  98  51  41  69  22]
 [ 35  33  43   0  78  52  19   9   4   2  94  90  72  54  11  73  91 100
   79  85  39  17   3  18  11  43   1  75  60  26]
 [ 30   5  10  78   0  12  97  59   6  40  36  53  78  13  27  65  26  16
   25   5  30  52  33  22   9   6  65   6  73  11]
 [ 13  70  98  85  48   0  89  60  55  80  34  72  39  61  33  29   9   8
   72  54  96  74  26  92  40  47  78   6  72  48]
 [  7  45  54   9  32  27   0  89  38  97   9  60  23  45   3  75   9   5
   14  31  78  31  70  20  96  67  84  14   8  22]
 [ 22  85  43  52  90  18  57   0   7   2  65  20  57  23  22   8  78  41
   38  44  72  38  91  91  44  15

In [7]:
tspApp.run()